In [1]:
from fuzzycar import Car
from pynq import Overlay , MMIO

In [2]:
ol = Overlay("car.bit")

In [3]:
car = Car(ol)

Configuring SPI controller...


In [4]:
gpio = MMIO(ol.axi_gpio_0.mmio.base_addr, 0x10000 , debug=False)

In [24]:
gpio.write(0x00, 0x00)

In [14]:
car.front.read_distance()

15

In [ ]:
from time import sleep, time
car.motor.set_pwm_time(50,1600)
sleep(0.5)
car.motor.set_pwm_time(50,1700)
sleep(0.5)
car.motor.set_pwm_time(50,1600)


In [ ]:
import numpy as np
import skfuzzy as fuzz
import skfuzzy.control as ctrl
from time import sleep, time
from IPython.display import clear_output

# ---------------------------
# Fuzzy Controller Definition
# ---------------------------

# Define fuzzy variables for sensor readings
left_sensor = ctrl.Antecedent(np.arange(6, 255, 1), 'Left Sensor')
front_sensor = ctrl.Antecedent(np.arange(6, 255, 1), 'Front Sensor')
right_sensor = ctrl.Antecedent(np.arange(6, 255, 1), 'Right Sensor')

# Define fuzzy variable for turning PWM in microseconds (600=full left, 1000=full right)
turning_pwm = ctrl.Consequent(np.arange(600, 1001, 1), 'Turning PWM')

# Membership functions for sensors (based on your earlier definitions)
left_sensor['Close Wall'] = fuzz.trapmf(left_sensor.universe, [0, 0, 70, 100])
left_sensor['Far Wall'] = fuzz.trapmf(left_sensor.universe, [70, 100, 125, 155])
left_sensor['Wall?'] = fuzz.trapmf(left_sensor.universe, [125, 155, 180, 180])

front_sensor['Close'] = fuzz.trapmf(front_sensor.universe, [0, 0, 22, 32])
front_sensor['Medium'] = fuzz.trapmf(front_sensor.universe, [22, 32, 42, 52])
front_sensor['Far'] = fuzz.trapmf(front_sensor.universe, [42, 52, 60, 60])

right_sensor['Close Wall'] = fuzz.trapmf(right_sensor.universe, [0, 0, 55, 85])
right_sensor['Far Wall'] = fuzz.trapmf(right_sensor.universe, [55, 85, 125, 155])
right_sensor['Wall?'] = fuzz.trapmf(right_sensor.universe, [125, 155, 180, 180])

# Membership functions for turning PWM
# Here 600 μs corresponds to full left and 1000 μs to full right, with 800 μs as the center (straight)
turning_pwm['Hard Left'] = fuzz.trimf(turning_pwm.universe, [600, 600, 700])
turning_pwm['Soft Left'] = fuzz.trimf(turning_pwm.universe, [600, 700, 800])
turning_pwm['Straight']  = fuzz.trimf(turning_pwm.universe, [700, 800, 900])
turning_pwm['Soft Right'] = fuzz.trimf(turning_pwm.universe, [800, 900, 1000])
turning_pwm['Hard Right'] = fuzz.trimf(turning_pwm.universe, [900, 1000, 1000])

# Define fuzzy rules mapping sensor conditions to turning commands.
rules = [
    ctrl.Rule(left_sensor['Close Wall'] & front_sensor['Medium'] & ~right_sensor['Close Wall'],
              turning_pwm['Soft Right']),
    ctrl.Rule(left_sensor['Close Wall'] & front_sensor['Close'] & ~right_sensor['Close Wall'],
              turning_pwm['Hard Right']),
    ctrl.Rule(~left_sensor['Close Wall'] & front_sensor['Medium'] & ~right_sensor['Wall?'],
              turning_pwm['Soft Left']),
    ctrl.Rule(~left_sensor['Close Wall'] & front_sensor['Close'] & ~right_sensor['Wall?'],
              turning_pwm['Hard Left']),
    ctrl.Rule(front_sensor['Far'] & (left_sensor['Close Wall'] | right_sensor['Close Wall']),
              turning_pwm['Straight']),
    ctrl.Rule(front_sensor['Far'] & ~(left_sensor['Close Wall'] | right_sensor['Close Wall']),
              turning_pwm['Straight']),
    ctrl.Rule(~left_sensor['Close Wall'] & front_sensor['Medium'] & right_sensor['Wall?'],
              turning_pwm['Soft Right']),
    ctrl.Rule(~left_sensor['Close Wall'] & front_sensor['Close'] & right_sensor['Wall?'],
              turning_pwm['Hard Right'])
]

# Create control system and simulation object
steering_ctrl = ctrl.ControlSystem(rules)
steering_sim = ctrl.ControlSystemSimulation(steering_ctrl)

def get_turning_pwm(left, front, right):
    """
    Compute the turning PWM (in microseconds) based on sensor readings using fuzzy logic.
    If no output is computed, returns a default value (800 μs for straight).
    """
    steering_sim.input['Left Sensor'] = left
    steering_sim.input['Front Sensor'] = front
    steering_sim.input['Right Sensor'] = right
    steering_sim.compute()

    # For debugging: print available output keys
    # print("Computed outputs:", steering_sim.output)
    
    if 'Turning PWM' in steering_sim.output:
        return steering_sim.output['Turning PWM']
    else:
        # If no output is computed, default to straight (800 μs)
        print("Warning: No fuzzy output computed. Defaulting to 800 μs (Straight).")
        return 800

# -------------------------------------------
# Example Integration with Your Hardware Code
# -------------------------------------------

# Assume sensor and steering objects (e.g., car.front, car.driver, car.passenger, car.steering) are defined
# Main loop: read sensors, compute fuzzy output, and update steering PWM.
while True:
    # Read distances (in inches) from the three sensors
    front_distance = car.front.read_distance()
    left_distance = car.driver.read_distance()
    right_distance = car.passenger.read_distance()    
    
    # Get the PWM value from fuzzy inference
    pwm_value = get_turning_pwm(left_distance, front_distance, right_distance)
    
    # Update the steering PWM; frequency is 50Hz as in your primitive example.
    car.steering.set_pwm_time(50, pwm_value)
    
    # Print the sensor readings and PWM for debugging
    print(f"Left: {left_distance}, Front: {front_distance}, Right: {right_distance}, PWM: {pwm_value:.2f}")
    clear_output(wait=True)    
    sleep(0.1)  # Adjust the loop delay as needed


In [ ]:
car.steering.set_pwm_time(50, 0)